# Module

## Meta
- learn
    - https://colah.github.io/posts/2015-08-Understanding-LSTMs/
    - https://distill.pub/2019/memorization-in-rnns/
- code
    - https://github.com/georgeyiasemis/Recurrent-Neural-Networks-from-scratch-using-PyTorch/
- changes
    - refined by me and help of DeepSeek

## Imports

In [165]:
import numpy  as np
import pandas as pd

import torch
import torch.nn  as nn

## Utils

In [166]:
def reset_parameters(model):
    std = 1.0 / np.sqrt(model.hidden_size)
    for w in model.parameters():
        w.data.uniform_(-std, std)

--------

# RNN

## Cell

### impl

In [167]:
class RNNCell(nn.Module):
    def __init__(self, input_size, hidden_size, bias=True, nonlinearity=torch.tanh):
        super().__init__()

        self.input_size   = input_size
        self.hidden_size  = hidden_size
        self.bias         = bias
        self.nonlinearity = nonlinearity

        self.x2h = nn.Linear(input_size,  hidden_size, bias=bias)
        self.h2h = nn.Linear(hidden_size, hidden_size, bias=bias)

    def forward(self, input, hx=None):
        # Inputs:
        #   input: (batch_size, input_size)
        #   hx:    (batch_size, hidden_size)
        # Output:
        #   hy:    (batch_size, hidden_size)
        
        batch_size = input.size(0)
        hx = torch.zeros(batch_size, self.hidden_size, device=input.device, dtype=input.dtype) if hx is None else hx
        hy = self.nonlinearity(self.x2h(input) + self.h2h(hx))
        return hy

### test

In [168]:
cell = RNNCell(input_size=10, hidden_size=20)

h = None  # Will be initialized on first call
for t in range(3):
    x_t = torch.randn(4, 10)
    h = cell(x_t, h)  # Pass previous hidden state
    print(f"Step {t+1}: h shape {h.shape}")

Step 1: h shape torch.Size([4, 20])
Step 2: h shape torch.Size([4, 20])
Step 3: h shape torch.Size([4, 20])


## Module

### impl

In [169]:
class RNNModule(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, bias, output_size, activation=torch.tanh):
        super().__init__()
        
        self.input_size  = input_size
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.bias        = bias
        self.output_size = output_size
        self.rnn_cell_list = nn.ModuleList([
              RNNCell(input_size,  hidden_size, bias, activation),
            *[RNNCell(hidden_size, hidden_size, bias, activation) for _ in range(1, num_layers)],
        ])
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, input, hx=None):
        outputs = []
        batch_size, seq_len, _ = input.size() # input: (batch_size, seq_len, input_size)

        hx = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=input.device, dtype=input.dtype) if hx is None else hx # Initialize hidden states
        hidden  = [hx[layer] for layer in range(self.num_layers)] # hx: (num_layers, batch_size, hidden_size) -> list of (batch_size, hidden_size)
        
        for t in range(seq_len):
            for layer in range(self.num_layers):
                x_t = input[:, t, :] if layer == 0 else hidden[layer - 1]  # (batch_size, input_size)
                hidden[layer] = self.rnn_cell_list[layer](x_t, hidden[layer])
            outputs.append(hidden[-1])
        
        out = self.fc(outputs[-1]) # (batch_size, hidden_size)
        # return out
        return out, torch.stack(hidden)  # ✅ Returns two values

### test

In [170]:
model = RNNModule(
    input_size=10,
    hidden_size=20,
    num_layers=2,
    bias=True,
    output_size=1,
    activation=torch.tanh
)

h = None

# Process sequence step by step
for t in range(5):
    x_t = torch.randn(4, 10)  # (batch_size, input_size)
    # ✅ Correct: Get both output AND new hidden state
    out, h = model(x_t.unsqueeze(1), h)  # unsqueeze to add seq_len dimension
    print(f"Step {t+1}: output shape {out.shape}, hidden shape {h.shape}")

Step 1: output shape torch.Size([4, 1]), hidden shape torch.Size([2, 4, 20])
Step 2: output shape torch.Size([4, 1]), hidden shape torch.Size([2, 4, 20])
Step 3: output shape torch.Size([4, 1]), hidden shape torch.Size([2, 4, 20])
Step 4: output shape torch.Size([4, 1]), hidden shape torch.Size([2, 4, 20])
Step 5: output shape torch.Size([4, 1]), hidden shape torch.Size([2, 4, 20])


# LSTM

## Cell

### impl

In [ ]:
class LSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size, bias=True):
        super().__init__()

        self.input_size  = input_size
        self.hidden_size = hidden_size
        self.bias        = bias

        self.xh = nn.Linear(input_size,  hidden_size * 4, bias=bias)
        self.hh = nn.Linear(hidden_size, hidden_size * 4, bias=bias)

    def forward(self, input, hx=None):
        # Inputs:
        #       input:  (batch_size, input_size)
        #       hx:     (batch_size, hidden_size)
        # Outputs:
        #       hy:     (batch_size, hidden_size)
        #       cy:     (batch_size, hidden_size)

        if hx is None:
            hx = torch.zeros(input.size(0), self.hidden_size, device=input.device, dtype=input.dtype)
            cx = torch.zeros(input.size(0), self.hidden_size, device=input.device, dtype=input.dtype)
            hx = (hx, cx)

        hx, cx = hx
        gates = self.xh(input) + self.hh(hx)

        # Get gates (i_t, f_t, g_t, o_t)
        input_gate, forget_gate, cell_gate, output_gate = gates.chunk(4, 1)
        # it's equivalent to
        # i_t = gates[:, 0            :1*hidden_size]
        # f_t = gates[:, 1*hidden_size:2*hidden_size]
        # g_t = gates[:, 2*hidden_size:3*hidden_size]
        # o_t = gates[:, 3*hidden_size:4*hidden_size]

        i_t = torch.sigmoid(input_gate)
        f_t = torch.sigmoid(forget_gate)
        g_t = torch.   tanh(cell_gate)
        o_t = torch.sigmoid(output_gate)

        cy = cx * f_t + i_t * g_t
        hy = o_t * torch.tanh(cy)
        return (hy, cy)

### test

In [172]:
cell = LSTMCell(input_size=10, hidden_size=20)

# Initialize hidden states
h = None
c = None

for t in range(3):
    x_t = torch.randn(4, 10)
    
    # ✅ Fix: Pass both h and c as a tuple
    if h is None:
        h, c = cell(x_t)  # First call auto-initializes
    else:
        h, c = cell(x_t, (h, c))  # Subsequent calls pass both
    
    print(f"Step {t+1}: h shape {h.shape}, c shape {c.shape}")

Step 1: h shape torch.Size([4, 20]), c shape torch.Size([4, 20])
Step 2: h shape torch.Size([4, 20]), c shape torch.Size([4, 20])
Step 3: h shape torch.Size([4, 20]), c shape torch.Size([4, 20])


## Module

### impl

In [173]:
class LSTMModule(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, bias, output_size):
        super().__init__()

        self.input_size  = input_size
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.bias        = bias
        self.output_size = output_size
        self.rnn_cell_list = nn.ModuleList([
              LSTMCell(self.input_size,  self.hidden_size, self.bias),
            *[LSTMCell(self.hidden_size, self.hidden_size, self.bias) for _ in range(1, self.num_layers)]
        ])
        self.fc = nn.Linear(self.hidden_size, self.output_size)

    def forward(self, input, hx=None):
        # Input:  (batch_size, seq_len, input_size)
        # Output: (batch_size, output_size)
        # hx:     tuple of (h0, c0) each (num_layers, batch_size, hidden_size)
        
        batch_size, seq_len, _ = input.size()
        
        # Initialize hidden states if not provided
        if hx is None:
            h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=input.device, dtype=input.dtype)
            c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=input.device, dtype=input.dtype)
        else:
            h0, c0 = hx
        
        # Initialize per-layer hidden states
        hidden = [(h0[layer], c0[layer]) for layer in range(self.num_layers)] # Each: (h, c) of shape (batch_size, hidden_size)
        
        # Process sequence
        outputs = []
        for t in range(seq_len):
            for layer in range(self.num_layers):
                # First layer takes input, others take previous layer's hidden state
                x_t = input[:, t, :] if layer == 0 else hidden[layer - 1][0]
                
                # Update hidden state for this layer
                h_prev, c_prev = hidden[layer]
                h_new,   c_new = self.rnn_cell_list[layer](x_t, (h_prev, c_prev))
                hidden[layer]  = (h_new, c_new)
            
            # Collect top layer's hidden state as output
            outputs.append(hidden[-1][0])
        
        out = self.fc(outputs[-1])  # Shape: (batch_size, output_size)
        
        # Return output and final hidden states (matching PyTorch API)
        h_n = torch.stack([h for h, c in hidden])  # (num_layers, batch_size, hidden_size)
        c_n = torch.stack([c for h, c in hidden])  # (num_layers, batch_size, hidden_size)
        
        return out, (h_n, c_n)

### test

In [174]:
# Create model
model = LSTMModule(
    input_size=10,
    hidden_size=20,
    num_layers=2,
    bias=True,
    output_size=1
)

# Test single forward
x = torch.randn(4, 3, 10)  # (batch_size, seq_len, input_size)
out, (h_n, c_n) = model(x)

print(f"Output shape: {out.shape}")      # (4, 1)
print(f"h_n shape: {h_n.shape}")         # (2, 4, 20)
print(f"c_n shape: {c_n.shape}")         # (2, 4, 20)

# Test multi-step with manual hidden state
h = None
for t in range(3):
    x_t = torch.randn(4, 10)
    out, (h, c) = model(x_t.unsqueeze(1), (h, c) if h is not None else None)
    print(f"Step {t+1}: output {out.shape}")

Output shape: torch.Size([4, 1])
h_n shape: torch.Size([2, 4, 20])
c_n shape: torch.Size([2, 4, 20])
Step 1: output torch.Size([4, 1])
Step 2: output torch.Size([4, 1])
Step 3: output torch.Size([4, 1])


# GRU

## Cell

### impl

In [175]:
class GRUCell(nn.Module):
    def __init__(self, input_size, hidden_size, bias=True):
        super().__init__()
        
        self.input_size  = input_size
        self.hidden_size = hidden_size
        self.bias        = bias
        
        # Combined linear layers for efficiency
        self.x2h = nn.Linear(input_size, 3 * hidden_size, bias=bias)
        self.h2h = nn.Linear(hidden_size, 3 * hidden_size, bias=bias)

    def forward(self, input, hx=None):
        # Inputs:
        #   input: (batch_size, input_size)
        #   hx:    (batch_size, hidden_size)
        # Output:
        #   hy:    (batch_size, hidden_size)
        
        batch_size = input.size(0)
        hx = torch.zeros(batch_size, self.hidden_size, device=input.device, dtype=input.dtype) if hx is None else hx
        
        # Project input and hidden state
        x_t = self.x2h(input)  # (batch_size, 3 * hidden_size)
        h_t = self.h2h(hx)     # (batch_size, 3 * hidden_size)
        
        # Split into reset, update, and candidate gates
        x_reset, x_update, x_candidate = x_t.chunk(3, 1)
        h_reset, h_update, h_candidate = h_t.chunk(3, 1)
        
        # Apply activations
        reset_gate     = torch.sigmoid(x_reset + h_reset)   # r_t
        update_gate    = torch.sigmoid(x_update + h_update) # z_t
        candidate_gate = torch.tanh(x_candidate + reset_gate * h_candidate)  # n_t
        
        # Update hidden state
        hy = update_gate * hx + (1 - update_gate) * candidate_gate
        
        return hy

### test

In [176]:
cell = GRUCell(input_size=10, hidden_size=20)

h = None  # Will be initialized on first call
for t in range(3):
    x_t = torch.randn(4, 10)
    h = cell(x_t, h)  # Pass previous hidden state
    print(f"Step {t+1}: h shape {h.shape}")

Step 1: h shape torch.Size([4, 20])
Step 2: h shape torch.Size([4, 20])
Step 3: h shape torch.Size([4, 20])


## Module

### impl

In [ ]:
class GRUModule(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, bias, output_size):
        super().__init__()

        self.input_size    = input_size
        self.hidden_size   = hidden_size
        self.num_layers    = num_layers
        self.bias          = bias
        self.output_size   = output_size
        self.rnn_cell_list = nn.ModuleList([
              GRUCell(self.input_size,  self.hidden_size, self.bias)
            *[GRUCell(self.hidden_size, self.hidden_size, self.bias) for _ in range(1, self.num_layers)]
        ])
        self.fc = nn.Linear(self.hidden_size, self.output_size)

    def forward(self, input, hx=None):
        # Input:  (batch_size, seq_len, input_size)
        # Output: (batch_size, output_size)
        # hx:     (num_layers, batch_size, hidden_size)
        
        batch_size, seq_len, _ = input.size()
        
        # Initialize hidden state if not provided
        hx = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=input.device, dtype=input.dtype) if hx is None else hx
        
        # Initialize per-layer hidden states
        hidden = [hx[layer] for layer in range(self.num_layers)]
        
        # Process sequence
        outputs = []
        for t in range(seq_len):
            for layer in range(self.num_layers):
                # First layer takes input, others take previous layer's hidden state
                x_t =  input[:, t, :] if layer == 0 else hidden[layer - 1]  # Previous layer's hidden state
                # Update hidden state for this layer
                hidden[layer] = self.rnn_cell_list[layer](x_t, hidden[layer])
            # Collect top layer's hidden state as output
            outputs.append(hidden[-1])
        
        # Final output
        out = self.fc(outputs[-1])  # Shape: (batch_size, output_size)
        
        # Return output and final hidden state (matching PyTorch API)
        h_n = torch.stack(hidden)  # (num_layers, batch_size, hidden_size)
        
        return out, h_n

### test

In [178]:
# Create model
model = GRUModule(
    input_size=10,
    hidden_size=20,
    num_layers=2,
    bias=True,
    output_size=1
)

# Test single forward
x = torch.randn(4, 3, 10)  # (batch_size, seq_len, input_size)
out, h_n = model(x)

print(f"Output shape: {out.shape}")  # (4, 1)
print(f"h_n shape: {h_n.shape}")     # (2, 4, 20)

# Test multi-step with manual hidden state
h = None
for t in range(3):
    x_t = torch.randn(4, 10)
    out, h = model(x_t.unsqueeze(1), h)
    print(f"Step {t+1}: output {out.shape}, hidden {h.shape}")

Output shape: torch.Size([4, 1])
h_n shape: torch.Size([2, 4, 20])
Step 1: output torch.Size([4, 1]), hidden torch.Size([2, 4, 20])
Step 2: output torch.Size([4, 1]), hidden torch.Size([2, 4, 20])
Step 3: output torch.Size([4, 1]), hidden torch.Size([2, 4, 20])


-----